In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, FormatStrFormatter, ScalarFormatter
RESULTS_DIR = Path("../results")
FIG_DIR = Path("./images")
FIG_DIR.mkdir(parents=True, exist_ok=True)

TEXT_WIDTH = 7.0
COLUMN_WIDTH = 3.35

DATASET_ORDER = ["20ng", "ag_news", "dbpedia14"]
DATASET_LABELS = {
    "20ng": "20NG",
    "ag_news": "AG News",
    "dbpedia14": "DBpedia14",
}

MODEL_LABELS = {
    "AttentiveTopicModel": "AARTM",
    "AttentiveTopicModelNoNWT": "AARTM-no-N",
    "ContextTopicModel": "CARTM",
    "LDA": "LDA",
    "NMF": "NMF",
    "BERTopic": "BERTopic",
    "CombinedTM": "CTM",
    "BTM": "BTM",
    "BigARTM": "BigARTM",
    "ContextualTop2Vec": "C-Top2Vec",
}

MODEL_ORDER = [
    "AARTM",
    "AARTM-no-N",
    "LDA",
    "NMF",
    "BigARTM",
    "BTM",
    "BERTopic",
    "CTM",
    "C-Top2Vec",
]

MODEL_PALETTE = {
    "AARTM": "#0072B2",
    "AARTM-no-N": "#56B4E9",
    "LDA": "#D55E00",
    "NMF": "#CC79A7",
    "BigARTM": "#E69F00",
    "BTM": "#B79F00",
    "BERTopic": "#009E73",
    "CTM": "#9467BD",
    "C-Top2Vec": "#8C564B",
    "CARTM": "#666666",
}

MODEL_MARKERS = {
    "AARTM": "o",
    "AARTM-no-N": "s",
    "LDA": "^",
    "NMF": "D",
    "BigARTM": "P",
    "BTM": "X",
    "BERTopic": "v",
    "CTM": "<",
    "C-Top2Vec": ">",
}

DATASET_PALETTE = {
    "20NG": "#0072B2",
    "AG News": "#D55E00",
    "DBpedia14": "#009E73",
}

METRIC_LABELS = {
    "npmi_10": "NPMI@10",
    "c_v_10": r"$C_v$@10",
    "topic_diversity_25": "TD@25",
    "macro_f1": "Macro-F1",
    "accuracy": "Accuracy",
    "train_time_sec": "Time (s)",
    "perplexity_test": "Perplexity",
    "normalized_perplexity": "Norm.\nperplexity",
    "boundary_mae": "MAE",
    "boundary_hit@5": "Hit@5",
    "boundary_hit@10": "Hit@10",
}

def set_paper_style():
    sns.set_theme(style="whitegrid", context="paper")

    mpl.rcParams.update({
        "figure.dpi": 150,
        "savefig.dpi": 600,
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset": "stix",

        "font.size": 8,
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "legend.title_fontsize": 7,

        "axes.linewidth": 0.6,
        "grid.linewidth": 0.35,
        "lines.linewidth": 1.3,
        "lines.markersize": 3.8,
        "patch.linewidth": 0.45,

        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",

        "axes.spines.top": False,
        "axes.spines.right": False,
    })

set_paper_style()

def save_figure(fig, name: str):
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=600)

In [ ]:
def load_result_frames(result_subdir: str, filename: str, datasets=DATASET_ORDER) -> pd.DataFrame:
    frames = []
    for dataset in datasets:
        path = RESULTS_DIR / result_subdir / dataset / filename
        if not path.exists():
            warnings.warn(f"Missing result file: {path}")
            continue
        frame = pd.read_csv(path)
        frames.append(frame)

    if not frames:
        raise FileNotFoundError(f"No files found for {result_subdir}/{filename}")

    return pd.concat(frames, ignore_index=True)


def standardize_dataset_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "dataset" in df.columns:
        df["dataset"] = df["dataset"].replace(DATASET_LABELS)
    return df


def standardize_model_names(df: pd.DataFrame, col: str = "model") -> pd.DataFrame:
    df = df.copy()
    if col in df.columns:
        df[col] = df[col].replace(MODEL_LABELS)
    return df


def safe_drop(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    return df.drop([c for c in cols if c in df.columns], axis=1)


def available_order(values, order):
    values = set(values)
    return [x for x in order if x in values]


def require_columns(df: pd.DataFrame, cols: list[str], context: str = ""):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {context}: {missing}")

# Main table

In [ ]:
main_df = load_result_frames(
    "main_table",
    "main_table_summary.csv",
    datasets=["20ng", "ag_news", "dbpedia14"],
)

main_df = standardize_dataset_names(main_df)
main_df = standardize_model_names(main_df, "model")

main_df = safe_drop(
    main_df,
    ["accuracy_mean", "accuracy_std", "seed_mean", "seed_std"],
)

for col in main_df.columns:
    if col not in ["dataset", "model"]:
        main_df[col] = pd.to_numeric(main_df[col], errors="coerce")

main_metrics = [
    ("npmi_10", METRIC_LABELS["npmi_10"]),
    ("c_v_10", METRIC_LABELS["c_v_10"]),
    ("topic_diversity_25", METRIC_LABELS["topic_diversity_25"]),
    ("macro_f1", METRIC_LABELS["macro_f1"]),
    ("train_time_sec", METRIC_LABELS["train_time_sec"]),
]

main_metrics = [
    (m, label)
    for m, label in main_metrics
    if f"{m}_mean" in main_df.columns
]

datasets = available_order(main_df["dataset"].unique(), list(DATASET_LABELS.values()))
models = available_order(main_df["model"].unique(), MODEL_ORDER)

n_rows = len(datasets)
n_cols = len(main_metrics)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(TEXT_WIDTH, 1.05 * n_rows),
    sharex=True,
    constrained_layout=False,
)

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)
if n_cols == 1:
    axes = np.expand_dims(axes, axis=1)

x = np.arange(len(models))
bar_width = 0.78

for i, dataset in enumerate(datasets):
    for j, (metric, metric_label) in enumerate(main_metrics):
        ax = axes[i, j]

        sub = (
            main_df[main_df["dataset"] == dataset]
            .set_index("model")
            .reindex(models)
        )

        means = sub[f"{metric}_mean"].to_numpy(dtype=float)
        stds = sub.get(f"{metric}_std", pd.Series(0.0, index=sub.index)).to_numpy(dtype=float)
        colors = [MODEL_PALETTE[m] for m in models]

        ax.bar(
            x,
            means,
            yerr=stds,
            width=bar_width,
            color=colors,
            edgecolor="black",
            linewidth=0.35,
            error_kw={
                "elinewidth": 0.55,
                "capsize": 1.5,
                "capthick": 0.55,
            },
        )

        if metric == "npmi_10":
            ax.axhline(0.0, color="black", linewidth=0.5)

        if metric == "train_time_sec":
            positive = means[np.isfinite(means) & (means > 0)]
            if len(positive) > 0:
                ax.set_yscale("log")

        if metric in {"topic_diversity_25", "macro_f1"}:
            ax.set_ylim(0, 1.05)

        if metric in {"c_v_10"}:
            ax.set_ylim(0.45, 0.8)

        if i == 0:
            ax.set_title(metric_label, pad=4)

        if j == 0:
            ax.set_ylabel(dataset, fontweight="bold")
        else:
            ax.set_ylabel("")

        ax.set_xticks(x)
        ax.set_xticklabels([])
        ax.tick_params(axis="x", length=0)
        ax.grid(axis="y", alpha=0.35)

handles = [
    Patch(
        facecolor=MODEL_PALETTE[m],
        edgecolor="black",
        linewidth=0.35,
        label=m,
    )
    for m in models
]

fig.legend(
    handles=handles,
    labels=models,
    loc="upper center",
    ncol=min(len(models), 5),
    frameon=False,
    bbox_to_anchor=(0.5, 1.03),
)

fig.tight_layout(rect=[0, 0, 1, 0.94])
save_figure(fig, "fig_main_results")
plt.show()

# Effective context ablation

In [ ]:
ctx_df = load_result_frames(
    "ablation/context_length",
    "ablation_summary.csv",
    datasets=DATASET_ORDER,
)

ctx_df = standardize_dataset_names(ctx_df)

if "model_variant" in ctx_df.columns:
    ctx_df = ctx_df[ctx_df["model_variant"] == "full"]

ctx_df["C"] = ctx_df["ctx_len"].astype(int)
ctx_df["effective_radius"] = np.minimum(ctx_df["C"], 1.0 / ctx_df["gamma"])

for col in ctx_df.columns:
    if col not in ["dataset", "model_variant"]:
        ctx_df[col] = pd.to_numeric(ctx_df[col], errors="coerce")

ctx_df["normalized_perplexity_mean"] = (
    ctx_df.groupby("dataset")["perplexity_test_mean"]
    .transform(lambda x: x / x.max())
)

metrics = [
    ("npmi_10_mean", METRIC_LABELS["npmi_10"]),
    ("c_v_10_mean", METRIC_LABELS["c_v_10"]),
    ("normalized_perplexity_mean", METRIC_LABELS["normalized_perplexity"]),
]

plot_df = (
    ctx_df
    .groupby(["dataset", "effective_radius"], as_index=False)
    [[m[0] for m in metrics]]
    .mean()
)

long_df = plot_df.melt(
    id_vars=["dataset", "effective_radius"],
    value_vars=[m[0] for m in metrics],
    var_name="metric",
    value_name="value",
)

metric_name_map = dict(metrics)
long_df["metric"] = long_df["metric"].map(metric_name_map)

fig, axes = plt.subplots(
    len(metrics),
    1,
    figsize=(COLUMN_WIDTH, 2.6),
    sharex=True,
)

if len(metrics) == 1:
    axes = [axes]

for ax, (_, metric_label) in zip(axes, metrics):
    sub_metric = long_df[long_df["metric"] == metric_label]

    for dataset in available_order(sub_metric["dataset"].unique(), list(DATASET_LABELS.values())):
        g = sub_metric[sub_metric["dataset"] == dataset].sort_values("effective_radius")

        ax.plot(
            g["effective_radius"],
            g["value"],
            label=dataset,
            color=DATASET_PALETTE[dataset],
            marker="o",
            linewidth=1.3,
            markersize=3.4,
        )

    ax.set_ylabel(metric_label)
    ax.set_xscale("log")
    ax.grid(True, which="both", alpha=0.35)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))

axes[-1].set_xlabel(r"Effective context radius $\min(C, 1/\gamma)$")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=min(len(labels), 3),
    frameon=False,
    bbox_to_anchor=(0.5, 1.02),
)

fig.tight_layout(rect=[0, 0, 1, 0.95])
save_figure(fig, "fig_context_radius_ablation")
plt.show()

# Self-aware context ablation

In [ ]:
sa_df = load_result_frames(
    "ablation/self_aware",
    "ablation_summary.csv",
    datasets=DATASET_ORDER,
)

sa_df = standardize_dataset_names(sa_df)

if "model_variant" in sa_df.columns:
    sa_df = sa_df[sa_df["model_variant"] == "full"]

sa_df["C"] = sa_df["ctx_len"].astype(int)

for col in sa_df.columns:
    if col not in ["dataset", "self_aware_context", "model_variant"]:
        sa_df[col] = pd.to_numeric(sa_df[col], errors="coerce")

PAPER_C = 100
PAPER_GAMMA = 0.01

sub = sa_df[(sa_df["C"] == PAPER_C) & (np.isclose(sa_df["gamma"], PAPER_GAMMA))].copy()

if sub.empty:
    raise ValueError(f"No rows found for C={PAPER_C}, gamma={PAPER_GAMMA}")

datasets = available_order(sub["dataset"].unique(), list(DATASET_LABELS.values()))

metrics = [
    ("macro_f1", METRIC_LABELS["macro_f1"]),
    ("c_v_10", METRIC_LABELS["c_v_10"]),
    ("topic_diversity_25", METRIC_LABELS["topic_diversity_25"]),
    ("perplexity_test", METRIC_LABELS["perplexity_test"]),
]

setting_labels = {
    False: "Self-excluding",
    True: "Self-aware",
}

setting_colors = {
    False: "#999999",
    True: MODEL_PALETTE["AARTM"],
}

fig, axes = plt.subplots(
    2,
    2,
    figsize=(COLUMN_WIDTH, 2.6),
    sharex=True,
)

axes = axes.ravel()
x = np.arange(len(datasets))
width = 0.36

for ax, (metric, metric_label) in zip(axes, metrics):
    for k, self_aware in enumerate([False, True]):
        means, stds = [], []

        for dataset in datasets:
            row = sub[
                (sub["dataset"] == dataset)
                & (sub["self_aware_context"] == self_aware)
            ]

            if len(row) != 1:
                raise ValueError(
                    f"Expected one row for dataset={dataset}, "
                    f"C={PAPER_C}, gamma={PAPER_GAMMA}, self_aware={self_aware}; got {len(row)}."
                )

            means.append(row[f"{metric}_mean"].iloc[0])
            stds.append(row.get(f"{metric}_std", pd.Series([0.0])).iloc[0])

        offset = (k - 0.5) * width

        ax.bar(
            x + offset,
            means,
            yerr=stds,
            width=width,
            color=setting_colors[self_aware],
            edgecolor="black",
            linewidth=0.4,
            error_kw={
                "elinewidth": 0.55,
                "capsize": 1.5,
                "capthick": 0.55,
            },
            label=setting_labels[self_aware],
        )

    ax.set_title(metric_label)
    ax.grid(axis="y", alpha=0.35)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))

    if metric in {"macro_f1", "topic_diversity_25"}:
        ax.set_ylim(0, 1.05)
    if metric == "perplexity_test":
        ax.set_yscale("log")

for ax in axes[2:]:
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=20, ha="right")

handles = [
    Patch(facecolor=setting_colors[v], edgecolor="black", linewidth=0.4, label=setting_labels[v])
    for v in [False, True]
]

fig.legend(
    handles=handles,
    loc="upper center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 1.03),
)

fig.tight_layout(rect=[0, 0, 1, 0.99])
save_figure(fig, "fig_self_aware_ablation")
plt.show()

# Attention passes ablation

In [ ]:
attn_df = load_result_frames(
    "ablation/attn_passes",
    "ablation_summary.csv",
    datasets=DATASET_ORDER,
)

attn_df = standardize_dataset_names(attn_df)
attn_df

In [ ]:
attn_df = load_result_frames(
    "ablation/attn_passes",
    "ablation_summary.csv",
    datasets=DATASET_ORDER,
)

attn_df = standardize_dataset_names(attn_df)

if "model_variant" in attn_df.columns:
    attn_df = attn_df[attn_df["model_variant"] == "full"]

if "self_aware_context" in attn_df.columns:
    attn_df = attn_df[attn_df["self_aware_context"] == False]

# If this directory contains multiple gamma values, choose the paper gamma.
if "gamma" in attn_df.columns and attn_df["gamma"].nunique() > 1:
    PAPER_GAMMA = 0.1
    attn_df = attn_df[np.isclose(attn_df["gamma"], PAPER_GAMMA)]

attn_df["C"] = attn_df["ctx_len"].astype(int)
attn_df["L"] = attn_df["num_attn_passes"].astype(int)

for col in attn_df.columns:
    if col not in ["dataset", "model_variant", "self_aware_context"]:
        attn_df[col] = pd.to_numeric(attn_df[col], errors="coerce")

datasets = available_order(attn_df["dataset"].unique(), list(DATASET_LABELS.values()))
context_values = sorted(attn_df["C"].unique())

metrics = [
    ("macro_f1", METRIC_LABELS["macro_f1"]),
    ("topic_diversity_25", METRIC_LABELS["topic_diversity_25"]),
    ("npmi_10", METRIC_LABELS["npmi_10"]),
    ("c_v_10", METRIC_LABELS["c_v_10"]),
]

context_palette = dict(zip(context_values, sns.color_palette("viridis", len(context_values))))

fig, axes = plt.subplots(
    len(metrics),
    len(datasets),
    figsize=(COLUMN_WIDTH, 4),
    sharex=True,
    constrained_layout=False,
)

if len(datasets) == 1:
    axes = axes[None, :]
if len(metrics) == 1:
    axes = axes[:, None]

for row_idx, (metric, metric_label) in enumerate(metrics):
    for col_idx, dataset in enumerate(datasets):

        ax = axes[row_idx, col_idx]

        sub_dataset = attn_df[attn_df["dataset"] == dataset]

        for C, group in sub_dataset.groupby("C"):
            group = group.sort_values("L")

            x = group["L"].to_numpy()
            y = group[f"{metric}_mean"].to_numpy()
            yerr = group.get(
                f"{metric}_std",
                pd.Series(0.0, index=group.index)
            ).to_numpy()

            ax.plot(
                x,
                y,
                marker="o",
                linewidth=1.1,
                markersize=2.8,
                color=context_palette[C],
                label=fr"$C={C}$",
            )

            ax.fill_between(
                x,
                y - yerr,
                y + yerr,
                color=context_palette[C],
                alpha=0.15,
                linewidth=0,
            )

        # top row: dataset titles
        if row_idx == 0:
            ax.set_title(dataset)

        # left column: metric labels
        if col_idx == 0:
            ax.set_ylabel(metric_label, fontweight="bold")

        # bottom row: x-axis labels
        if row_idx == len(metrics) - 1:
            ax.set_xlabel(r"$L$")

        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.grid(True, alpha=0.35)

handles, labels = axes[0, -1].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    title="Context",
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=min(len(labels), 5),
    frameon=False,
)

fig.tight_layout(rect=[0, 0, 1, 0.96])
save_figure(fig, "fig_attention_passes_ablation")
plt.show()

# No N_wt

In [ ]:
no_nwt_df = load_result_frames(
    "ablation/no_ntw",
    "ablation_summary.csv",
    datasets=DATASET_ORDER,
)
full_df = load_result_frames(
    "ablation/self_aware",
    "ablation_summary.csv",
    datasets=DATASET_ORDER,
)

ab_df = pd.concat([no_nwt_df, full_df], ignore_index=True)
ab_df = standardize_dataset_names(ab_df)

if "self_aware_context" in ab_df.columns:
    ab_df = ab_df[ab_df["self_aware_context"] == False]

PAPER_C = 100
PAPER_GAMMA = 0.01

ab_df = ab_df[
    (ab_df["ctx_len"].astype(int) == PAPER_C)
    & (np.isclose(ab_df["gamma"], PAPER_GAMMA))
].copy()

ab_df["model_variant"] = ab_df["model_variant"].replace({
    "full": "AARTM",
    "no_nwt": "AARTM-no-N",
})

for col in ab_df.columns:
    if col not in ["dataset", "model_variant"]:
        ab_df[col] = pd.to_numeric(ab_df[col], errors="coerce")

datasets = available_order(ab_df["dataset"].unique(), list(DATASET_LABELS.values()))
models = ["AARTM", "AARTM-no-N"]

metrics = [
    ("macro_f1", METRIC_LABELS["macro_f1"]),
    ("topic_diversity_25", METRIC_LABELS["topic_diversity_25"]),
    ("npmi_10", METRIC_LABELS["npmi_10"]),
    ("c_v_10", METRIC_LABELS["c_v_10"]),
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(COLUMN_WIDTH, 2.6),
    sharex=True,
)

axes = axes.ravel()
x = np.arange(len(datasets))
width = 0.36

for ax, (metric, metric_label) in zip(axes, metrics):
    for k, model in enumerate(models):
        means, stds = [], []

        for dataset in datasets:
            row = ab_df[
                (ab_df["dataset"] == dataset)
                & (ab_df["model_variant"] == model)
            ]

            if len(row) != 1:
                raise ValueError(f"Expected one row for dataset={dataset}, model={model}; got {len(row)}")

            means.append(row[f"{metric}_mean"].iloc[0])
            stds.append(row.get(f"{metric}_std", pd.Series([0.0])).iloc[0])

        offset = (k - 0.5) * width

        ax.bar(
            x + offset,
            means,
            yerr=stds,
            width=width,
            color=MODEL_PALETTE[model],
            edgecolor="black",
            linewidth=0.4,
            error_kw={
                "elinewidth": 0.55,
                "capsize": 1.5,
                "capthick": 0.55,
            },
            label=model,
        )

    ax.set_title(metric_label)
    ax.grid(axis="y", alpha=0.35)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))

    if metric in {"macro_f1"}:
        ax.set_ylim(0.5, 1.05)
    if metric in {"topic_diversity_25"}:
        ax.set_ylim(0.6, 0.9)
    if metric == "npmi_10":
        ax.axhline(0, color="black", linewidth=0.5)
    if metric == "c_v_10":
        ax.set_ylim(0.6, 0.75)

for ax in axes[2:]:
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=20, ha="right")

# axes[0].set_xticks(x)
# axes[0].set_xticklabels([])

handles = [
    Patch(facecolor=MODEL_PALETTE[m], edgecolor="black", linewidth=0.4, label=m)
    for m in models
]

fig.legend(
    handles=handles,
    loc="upper center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 1.03),
)

fig.tight_layout(rect=[0, 0, 1, 0.99])
save_figure(fig, "fig_nwt_ablation")
plt.show()

# Length robustness

In [ ]:
def load_length_summary(datasets=DATASET_ORDER):
    frames = []

    for dataset in datasets:
        path = RESULTS_DIR / "length_robustness" / dataset / "length_robustness_summary.csv"
        if not path.exists():
            warnings.warn(f"Missing result file: {path}")
            continue

        df = pd.read_csv(path, header=[0, 1, 2])
        flat_cols = []
        for a, b, c in df.columns:
            if str(a).startswith("Unnamed") and str(b).startswith("Unnamed"):
                flat_cols.append(c)
            elif str(b).startswith("Unnamed"):
                flat_cols.append(a)
            else:
                flat_cols.append(f"{a}_{b}")
        df.columns = flat_cols
        df = df.rename(columns={
            "max_tokens_per_doc": "tokens",
            "macro_f1_mean": "f1_mean",
            "macro_f1_std": "f1_std",
        })
        frames.append(df)

    if not frames:
        raise FileNotFoundError("No length robustness files found.")

    return pd.concat(frames, ignore_index=True)


len_df = load_length_summary()
len_df = standardize_dataset_names(len_df)
len_df = standardize_model_names(len_df, "model")

len_df = len_df[~len_df["dataset"].isna() & ~len_df["f1_mean"].isna()].copy()
len_df = len_df[len_df["tokens"] > 1]

for col in ["tokens", "f1_mean", "f1_std"]:
    len_df[col] = pd.to_numeric(len_df[col], errors="coerce")

datasets = available_order(len_df["dataset"].unique(), list(DATASET_LABELS.values()))
models = available_order(
    len_df["model"].unique(),
    ["AARTM", "LDA", "NMF"],
)

fig, axes = plt.subplots(
    len(datasets),
    1,
    figsize=(COLUMN_WIDTH, 2.6),
    sharex=True,
)

if len(datasets) == 1:
    axes = [axes]

for ax, dataset in zip(axes, datasets):
    sub_dataset = len_df[len_df["dataset"] == dataset]

    for model in models:
        g = sub_dataset[sub_dataset["model"] == model].sort_values("tokens")
        if g.empty:
            continue

        x = g["tokens"].to_numpy()
        y = g["f1_mean"].to_numpy()
        yerr = g["f1_std"].to_numpy()

        ax.plot(
            x,
            y,
            label=model,
            color=MODEL_PALETTE[model],
            marker=MODEL_MARKERS[model],
            linewidth=1.3,
            markersize=3.5,
        )

        ax.fill_between(
            x,
            y - yerr,
            y + yerr,
            color=MODEL_PALETTE[model],
            alpha=0.14,
            linewidth=0,
        )

    ax.set_ylabel(dataset, fontweight="bold")
    # ax.set_ylim(0, 1.05)
    ax.set_xscale("log", base=2)
    ax.grid(True, which="both", alpha=0.35)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))

axes[-1].set_xlabel("Observed tokens per document")
axes[-1].set_xticks(sorted(len_df["tokens"].dropna().unique()))
axes[-1].get_xaxis().set_major_formatter(ScalarFormatter())

handles, labels = axes[-1].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=min(len(labels), 4),
    frameon=False,
    bbox_to_anchor=(0.5, 1.02),
)

fig.tight_layout(rect=[0, 0, 1, 0.95])
save_figure(fig, "fig_length_robustness")
plt.show()

# Boundary detection

In [ ]:
bd_df = load_result_frames(
    "boundary_detection",
    "boundary_detection_summary.csv",
    datasets=DATASET_ORDER,
)

bd_df = standardize_dataset_names(bd_df)
bd_df = standardize_model_names(bd_df, "model")

for col in bd_df.columns:
    if col not in ["dataset", "model"]:
        bd_df[col] = pd.to_numeric(bd_df[col], errors="coerce")

datasets = available_order(bd_df["dataset"].unique(), list(DATASET_LABELS.values()))
models = available_order(
    bd_df["model"].unique(),
    ["AARTM", "LDA", "NMF"],
)

metrics = [
    ("boundary_mae", METRIC_LABELS["boundary_mae"]),
    ("boundary_hit@10", METRIC_LABELS["boundary_hit@10"]),
]

fig, axes = plt.subplots(
    len(metrics),
    len(datasets),
    figsize=(COLUMN_WIDTH, 2),
    sharex=True,
)

if len(metrics) == 1:
    axes = axes[None, :]
if len(datasets) == 1:
    axes = axes[:, None]

x = np.arange(len(models))
width = 0.72

for row_idx, (metric, metric_label) in enumerate(metrics):
    for col_idx, dataset in enumerate(datasets):
        ax = axes[row_idx, col_idx]

        sub = (
            bd_df[bd_df["dataset"] == dataset]
            .set_index("model")
            .reindex(models)
        )

        means = sub[f"{metric}_mean"].to_numpy(dtype=float)
        stds = sub.get(f"{metric}_std", pd.Series(0.0, index=sub.index)).to_numpy(dtype=float)

        ax.bar(
            x,
            means,
            yerr=stds,
            width=width,
            color=[MODEL_PALETTE[m] for m in models],
            edgecolor="black",
            linewidth=0.4,
            error_kw={
                "elinewidth": 0.55,
                "capsize": 1.5,
                "capthick": 0.55,
            },
        )

        if row_idx == 0:
            ax.set_title(dataset)

        if col_idx == 0:
            ax.set_ylabel(metric_label)

        # if row_idx == len(metrics) - 1:
        #     ax.set_xticks(x)
        #     ax.set_xticklabels(models, rotation=35, ha="right")
        # else:
        ax.set_xticks(x)
        ax.set_xticklabels([])

        if metric.startswith("boundary_hit"):
            ax.set_ylim(0.5, 1)
        else:
            upper = np.nanmax(means + stds)
            ax.set_ylim(0, max(1.0, upper * 1.2))

        ax.grid(axis="y", alpha=0.35)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))

handles = [
    Patch(facecolor=MODEL_PALETTE[m], edgecolor="black", linewidth=0.4, label=m)
    for m in models
]

fig.legend(
    handles,
    models,
    loc="upper center",
    ncol=min(len(models), 4),
    frameon=False,
    bbox_to_anchor=(0.5, 1.03),
)

fig.tight_layout(rect=[0, 0, 1, 0.98])
save_figure(fig, "fig_boundary_detection")
plt.show()

# Benchmark

In [ ]:
benchmark_path = RESULTS_DIR / "benchmark" / "summary.csv"

if benchmark_path.exists():
    bench_df = pd.read_csv(benchmark_path)

    if "Unnamed: 0" in bench_df.columns:
        bench_df = bench_df.drop(columns=["Unnamed: 0"])

    bench_df["model"] = bench_df["model"].replace({
        "AttentiveTopicModel": "AARTM",
        "AttentiveTopicModelNoNWT": "AARTM-no-N",
    })

    models = available_order(bench_df["model"].unique(), ["AARTM", "AARTM-no-N"])

    fig, axes = plt.subplots(
        1,
        len(models),
        figsize=(COLUMN_WIDTH, 1.75),
        sharey=True,
    )

    if len(models) == 1:
        axes = [axes]

    vmin = bench_df["steady_mean_sec"].min()
    vmax = bench_df["steady_mean_sec"].max()

    for ax, model in zip(axes, models):
        sub = bench_df[bench_df["model"] == model].copy()

        mean_pivot = sub.pivot_table(
            index="ctx_len",
            columns="n_topics",
            values="steady_mean_sec",
            aggfunc="mean",
        ).sort_index()

        std_pivot = sub.pivot_table(
            index="ctx_len",
            columns="n_topics",
            values="steady_std",
            aggfunc="mean",
        ).reindex_like(mean_pivot)

        labels = mean_pivot.copy().astype(object)
        for r in mean_pivot.index:
            for c in mean_pivot.columns:
                m = mean_pivot.loc[r, c]
                s = std_pivot.loc[r, c]
                labels.loc[r, c] = "" if pd.isna(m) else f"{m:.2f}\n±{s:.2f}"

        sns.heatmap(
            mean_pivot,
            ax=ax,
            cmap="viridis",
            annot=labels,
            fmt="",
            cbar=(ax is axes[-1]),
            vmin=vmin,
            vmax=vmax,
            linewidths=0.3,
            linecolor="white",
            annot_kws={"fontsize": 6},
        )

        ax.set_title(model)
        ax.set_xlabel("Topics")
        ax.set_ylabel("Context size" if ax is axes[0] else "")

    fig.tight_layout()
    save_figure(fig, "fig_runtime_benchmark")
    plt.show()
else:
    print(f"Benchmark file not found: {benchmark_path}")